# Lecture 3 — Linear Classifiers

**Course:** UMich EECS 498-007 / 598-005 (Justin Johnson), Fall 2019, Lecture 3.  
**Merged with:** Stanford CS231n 2026 [Lecture 2](https://cs231n.stanford.edu/slides/2026/lecture_2.pdf) (second half: linear classifier, softmax) and [Lecture 3](https://cs231n.stanford.edu/slides/2026/lecture_3.pdf) (regularization; SVM vs. softmax).

> **About the sources.** UMich-specific content is based on knowledge of Justin Johnson's lecture (its slides couldn't be read directly); the CS231n 2026 slides were read in full. **[CS231n 2026]** marks content only there; **🔎** marks material beyond both.

### How to read this notebook
* Each **code cell does one thing**, with an explanation before it and **"What to notice"** after important results.
* Every function has a comment saying **what it computes, what goes in, and what comes out**.
* New ideas start with a **tiny example you can check by hand**. **✍️ Try it** boxes suggest experiments. Run cells **in order**.

**Why this lecture matters:** the linear classifier is the **Lego brick of neural networks**. Every fully connected layer, and the final classification layer of every CNN and Transformer, is a linear classifier. Today:
1. **What it computes**, seen three ways (algebraic, visual, geometric).
2. **How to measure how good it is**: loss functions (multiclass SVM and softmax).
3. **How to prefer simpler solutions**: regularization.

How to *find* the best weights (optimization) is Lecture 4; here we use a few lines of gradient descent as a preview.

| § | Topic | What you'll build |
|---|---|---|
| 1 | The parametric approach $f(x, W) = Wx + b$ | The slides' 4-pixel example (and a typo in it); the bias trick |
| 2 | Three viewpoints | Learned templates; the "two-headed horse"; decision regions |
| 3 | Hard cases | XOR, rings, multiple modes; fixing them with feature transforms |
| 4 | Loss functions; **multiclass SVM** | The slides' 3-image example, term by term; the lecture's questions in code |
| 5 | **Regularization** | L1 vs. L2 preferences; an overfit polynomial; λ on digits |
| 6 | **Softmax / cross-entropy** | The slides' 2.04, step by step; stable softmax; KL divergence |
| 7 | Gradients, and **gradient checking** | Loop and vectorized versions of both losses |
| 8 | SVM vs. softmax | How each reacts to changing scores |
| 9 | Training both on real digits | Accuracy; L1 makes weights sparse |
| 10 | PyTorch on CIFAR-10 | (Colab) |

### Setup

In [ ]:
import numpy as np                        # arrays and math
import matplotlib.pyplot as plt           # plots
from sklearn.datasets import load_digits  # the 8x8 handwritten digits
import math, time

np.set_printoptions(precision=4, suppress=True)

**Load and split the digits** (1,797 images of 8×8 pixels; Lecture 2). We scale pixels to [0, 1] and make three splits: **train** (to learn), **validation** (to choose hyperparameters), **test** (to report, once).

In [ ]:
digits = load_digits()
images = digits.images / 16.0                    # pixel values 0..16 -> 0..1
labels = digits.target
split = np.random.default_rng(0).permutation(len(images))
tr, va, te = split[:1000], split[1000:1300], split[1300:]
print(f"train {len(tr)}, validation {len(va)}, test {len(te)} images")

**Preprocessing, two steps:**
1. 🔎 **Subtract the mean training image.** Each pixel then measures "more or less ink than usual", centered at 0. This makes training better behaved (Lecture 4 explains why). Use the mean of the *training* set only, so no information leaks from validation/test data.
2. **Append a constant 1** to every image vector: the **bias trick** (§1.3). Each image becomes 64 + 1 = 65 numbers.

In [ ]:
mean_image = images[tr].mean(axis=0)            # (8, 8): the average training digit

def prep(idx, imgs=images):
    # Flatten the chosen images, subtract the mean training image, and append a constant-1 feature.
    X = (imgs[idx] - mean_image).reshape(len(idx), -1)       # (N, 64)
    return np.hstack([X, np.ones((len(idx), 1))])             # (N, 65)

X_tr, X_va, X_te = prep(tr), prep(va), prep(te)
y_tr, y_va, y_te = labels[tr], labels[va], labels[te]
print("training matrix:", X_tr.shape, "  last column (the bias feature):", X_tr[:3, -1])
plt.figure(figsize=(2, 2)); plt.imshow(mean_image, cmap="gray_r"); plt.title("mean training image", fontsize=8); plt.axis("off"); plt.show()

---
## 1. The parametric approach

Lecture 2's nearest neighbor **kept the entire training set** and compared every new image with all of it. A **parametric** model instead squeezes what it learned into a fixed set of numbers, the **parameters** (or weights) $W$, and throws the training data away. The linear classifier:
$$\underbrace{f(x, W)}_{10\text{ class scores}} = \underbrace{W}_{10\times3072}\ \underbrace{x}_{3072\times1} + \underbrace{b}_{10\times1}$$
for a CIFAR-10 image $x$ (32×32×3 = 3,072 numbers, stretched into a column).
* Each **row** of $W$ produces **one class's score**.
* The **bias** $b$ adds a fixed preference for each class, independent of the image (useful, e.g., if some classes are more common).
* The **prediction** is the class with the highest score.

How much smaller is that than nearest neighbor?

In [ ]:
D, C, N_train = 32 * 32 * 3, 10, 50_000
params = C * D + C
print(f"linear classifier on CIFAR-10: {C} x {D} weights + {C} biases = {params:,} parameters")
print(f"nearest neighbor stores the training set: {N_train * D:,} numbers  ({N_train * D / params:,.0f}x more)")

### 1.1 A tiny example: a 4-pixel image, 3 classes (cat / dog / ship)
From the slides: stretch the 2×2 image's pixels $(56, 231, 24, 2)$ into a column, multiply by $W$ (one row per class), and add $b$. Each class score is a **weighted sum of the pixels** plus that class's bias. Let's compute the cat score by hand first:
$$s_{\text{cat}} = 0.2\cdot56 + (-0.5)\cdot231 + 0.1\cdot24 + 2.0\cdot2 + 1.1$$

In [ ]:
x = np.array([56, 231, 24, 2], float)                # the flattened 2x2 image
W = np.array([[0.2, -0.5, 0.1, 2.0],                 # cat weights
              [1.5, 1.3, 2.1, 0.0],                  # dog weights
              [0.0, 0.25, 0.2, -0.3]])               # ship weights
b = np.array([1.1, 3.2, -1.2])

cat_by_hand = 0.2 * 56 + (-0.5) * 231 + 0.1 * 24 + 2.0 * 2 + 1.1
print(f"cat score by hand: {cat_by_hand:.2f}")

Now all three at once with a matrix–vector product:

In [ ]:
scores = W @ x + b
for name, s, s_no_bias in zip(["cat", "dog", "ship"], scores, W @ x):
    print(f"{name:5s}: W x + b = {s:8.2f}    (W x alone = {s_no_bias:8.2f})")
print("the slides show: cat -96.8, dog 437.9, ship 61.95")

**What to notice:**
* Cat and dog match the slides exactly.
* 🔎 **The slide's ship score, 61.95, is $Wx$ *without* the bias**; with it, $61.95 - 1.2 = 60.75$. A small typo, and a reminder to compute rather than copy.
* The classifier says **dog** by a huge margin, for what is supposed to be a cat. That's a bad classifier, and we need a way to *measure* how bad: a **loss function** (§4).

### 1.2 Conventions: $Wx$ vs. $XW$
Lectures write one image as a **column** $x$: $s = Wx + b$ with $W$ of shape $C\times D$. Code usually processes a **batch** of $N$ images stored as **rows** of a matrix $X$ ($N\times D$), and writes $S = XW + b$ with $W$ of shape $D\times C$, giving scores $S$ of shape $N\times C$. Same model, transposed. **The code in this notebook uses the batch convention.**

### 1.3 The bias trick
Append a constant 1 to $x$ and add $b$ as an extra column of $W$:
$$Wx + b = \begin{bmatrix}W & b\end{bmatrix}\begin{bmatrix}x\\1\end{bmatrix}$$
One matrix instead of two, which simplifies both math and code. (That's the column of ones we appended in `prep`.)

In [ ]:
W_aug = np.hstack([W, b[:, None]])           # W with b as an extra column: shape (3, 5)
x_aug = np.append(x, 1.0)                    # x with a 1 appended: shape (5,)
print("bias trick gives identical scores:", np.allclose(W_aug @ x_aug, W @ x + b))

🔎 **Predictions are linear in the image.** Ignoring the bias, scaling the image scales every score by the same factor: $f(cx) = W(cx) = c\,f(x)$. A darker image (smaller $c$) gives smaller, less confident scores, but the same ranking:

In [ ]:
print("scores of x      :", W @ x)
print("scores of 0.5 * x:", W @ (0.5 * x), " (exactly half)")

---
## 2. Three ways to look at a linear classifier

### 2.1 Algebraic viewpoint
$f(x, W) = Wx + b$: **each class score is a weighted sum of the pixels**. A positive weight on a pixel means "this pixel being bright is evidence *for* the class"; a negative weight means evidence *against*.

### 2.2 Visual viewpoint: template matching
Each **row of $W$** has exactly one weight per pixel, so we can **reshape it back into an image**: a **template** for that class. The score is a dot product between the image and the template, which is large when the image **matches** the template.

To see real templates we need a trained classifier. Training is Lecture 4's topic, so here are the two pieces as a preview, with comments:
* `softmax_loss`: measures how wrong the current scores are (§6 explains it in detail) and returns the gradient, the direction that would make it worse.
* `train_linear`: repeatedly takes a small random batch of training images and moves $W$ a small step **against** the gradient (gradient descent).

In [ ]:
def softmax_loss(W, X, y, reg):
    # Softmax (cross-entropy) loss with L2 regularization, and its gradient.
    # W: (D, C) weights, X: (N, D) images (rows), y: (N,) labels, reg: regularization strength. Returns (loss, dL/dW).
    N = X.shape[0]
    S = X @ W                                         # (N, C) scores
    S = S - S.max(axis=1, keepdims=True)              # subtract each row's max for numerical safety (Sec. 6.2)
    P = np.exp(S); P /= P.sum(axis=1, keepdims=True)  # (N, C) probabilities
    loss = -np.log(P[np.arange(N), y]).mean() + reg * np.sum(W * W)
    dS = P.copy(); dS[np.arange(N), y] -= 1; dS /= N  # d loss / d scores = probabilities - one-hot targets (Sec. 7)
    return loss, X.T @ dS + 2 * reg * W

def train_linear(loss_fn, X, y, reg=1e-3, lr=0.5, steps=1500, batch=128, seed=0):
    # Minibatch gradient descent on a linear classifier. Returns the learned W of shape (D, 10).
    r = np.random.default_rng(seed)
    W = 0.001 * r.normal(size=(X.shape[1], 10))       # start with small random weights
    for _ in range(steps):
        idx = r.integers(0, len(X), batch)            # a random minibatch
        _, dW = loss_fn(W, X[idx], y[idx], reg)
        W -= lr * dW                                  # step against the gradient
    return W

def accuracy(W, X, y):
    # Fraction of examples whose highest-scoring class is the correct one.
    return np.mean(np.argmax(X @ W, axis=1) == y)

**Train**, then look at the templates (column $c$ of our $W$, minus the bias row, reshaped to 8×8):

In [ ]:
W_soft = train_linear(softmax_loss, X_tr, y_tr)
print(f"softmax linear classifier: train accuracy {accuracy(W_soft, X_tr, y_tr):.1%}, validation {accuracy(W_soft, X_va, y_va):.1%}")

fig, ax = plt.subplots(1, 10, figsize=(14, 1.8))
for c in range(10):
    ax[c].imshow(W_soft[:64, c].reshape(8, 8), cmap="RdBu_r"); ax[c].set_title(str(c)); ax[c].axis("off")
plt.suptitle("learned templates (red = ink here is evidence FOR the class, blue = AGAINST)", fontsize=9); plt.show()

**What to notice:** each template looks like a blurry version of its digit: red where that digit has ink, blue where *other* digits have ink but this one doesn't. Let's confirm that a score really is a template match: the "3" score of an image equals the sum of (image × the "3" template):

In [ ]:
img_idx, c = 0, 3
template_match = np.sum(X_tr[img_idx, :64] * W_soft[:64, c]) + X_tr[img_idx, 64] * W_soft[64, c]   # pixels x template + bias
print(f"score for class {c}: from the classifier {(X_tr[img_idx] @ W_soft)[c]:.4f}, as a template match {template_match:.4f}")

On CIFAR-10, the templates look similar: the car template is a blurry red car (many CIFAR cars are red), and the plane and ship templates are mostly blue (sky and water). **A linear classifier uses context**: background color counts as evidence, which helps on average but fails for a red car parked in front of the sea.

**The key limitation: one template per class.** When a class has several **modes** (looks), the template must **average** them. CIFAR-10's **horse template has two heads**, because horses face both left and right. We can reproduce the effect: train on the digits **plus their mirror images** (as if digits could face either way).

**Step 1: build the mirrored training set.** `images[:, :, ::-1]` reverses each image's columns (a left-right flip).

In [ ]:
flipped = images[:, :, ::-1]
X_tr_mirror = np.vstack([X_tr, prep(tr, flipped)])     # originals + mirror images
y_tr_mirror = np.r_[y_tr, y_tr]                         # same labels
print("training set with mirror images:", X_tr_mirror.shape)

**Step 2: train on it and compare templates** for some asymmetric digits.

In [ ]:
W_mirror = train_linear(softmax_loss, X_tr_mirror, y_tr_mirror)
fig, ax = plt.subplots(2, 4, figsize=(8, 4))
for j, c in enumerate([2, 3, 4, 7]):
    ax[0, j].imshow(W_soft[:64, c].reshape(8, 8), cmap="RdBu_r"); ax[0, j].set_title(f"'{c}': normal data", fontsize=8)
    ax[1, j].imshow(W_mirror[:64, c].reshape(8, 8), cmap="RdBu_r"); ax[1, j].set_title(f"'{c}': + mirror images", fontsize=8)
for a in ax.ravel(): a.axis("off")
plt.show()

**Step 3: measure.** How left-right symmetric is a template? Correlate it with its own mirror image: 1 = perfectly symmetric. And how accurate is each classifier on ordinary (unflipped) digits?

In [ ]:
def symmetry(Wm):
    # Average correlation between each class template and its left-right mirror image (1.0 = perfectly symmetric).
    corrs = []
    for c in range(10):
        t = Wm[:64, c].reshape(8, 8)
        corrs.append(np.corrcoef(t.ravel(), t[:, ::-1].ravel())[0, 1])
    return np.mean(corrs)

print(f"template symmetry: normal training {symmetry(W_soft):.2f},  with mirror images {symmetry(W_mirror):.2f}")
print(f"validation accuracy on normal digits: normal training {accuracy(W_soft, X_va, y_va):.1%},  "
      f"with mirror images {accuracy(W_mirror, X_va, y_va):.1%}")

**What to notice:** with both orientations in the data, every template becomes a **symmetric blend** (the digits' "two-headed horse"), and accuracy on ordinary digits drops: one template can't represent "this shape OR its mirror image" without blurring both. A neural network with a hidden layer can (Lecture 5).

### 2.3 Geometric viewpoint: hyperplanes
Think of each image as a **point** in a high-dimensional space (one axis per pixel). Each class score $s_c = w_c^\top x + b_c$ is a **linear function** of that point:
* It's **zero on a hyperplane** (a line in 2-D), positive on one side, and grows in the direction of $w_c$.
* The boundary between classes $i$ and $j$ is where their scores are equal, $(w_i - w_j)^\top x + (b_i - b_j) = 0$: another hyperplane.

So **a linear classifier cuts space into regions with flat walls**. In 2-D we can draw it. **Step 1: a 2-D dataset with 3 classes.**

In [ ]:
pts_rng = np.random.default_rng(1)
centers = np.array([[-2, 0], [2, 1], [0, 2.5]])
P2 = np.vstack([c + pts_rng.normal(0, 0.7, (60, 2)) for c in centers])   # 60 points around each center
Y2 = np.repeat([0, 1, 2], 60)
X2 = np.hstack([P2, np.ones((len(P2), 1))])                              # bias trick: (x1, x2, 1)
print("2-D data:", X2.shape)

**Step 2: train** with full-batch gradient descent (same loss and gradient as above, written compactly):

In [ ]:
def train_small(X, y, n_classes, steps=3000, lr=0.5, reg=1e-4):
    # Full-batch gradient descent on the softmax loss. Returns W of shape (D, n_classes).
    W = np.zeros((X.shape[1], n_classes))
    for _ in range(steps):
        S = X @ W; S -= S.max(1, keepdims=True)
        P = np.exp(S); P /= P.sum(1, keepdims=True)
        P[np.arange(len(y)), y] -= 1                        # probabilities - one-hot targets
        W -= lr * (X.T @ P / len(y) + 2 * reg * W)
    return W

W2 = train_small(X2, Y2, 3)
print(f"2-D training accuracy: {np.mean(np.argmax(X2 @ W2, 1) == Y2):.1%}")

**Step 3: draw it**: the colored regions show which class wins at each point; the dashed lines are where each class's score equals 0; the arrows show each $w_c$ (the direction in which that class's score grows fastest).

In [ ]:
gx, gy = np.meshgrid(np.linspace(-5, 5, 300), np.linspace(-3, 5.5, 300))
G = np.c_[gx.ravel(), gy.ravel(), np.ones(gx.size)]                       # every plot pixel, with the bias feature
regions = np.argmax(G @ W2, axis=1).reshape(gx.shape)
cols = ["#2171b5", "#cb181d", "#238b45"]
plt.figure(figsize=(6.5, 5))
plt.contourf(gx, gy, regions, levels=[-0.5, 0.5, 1.5, 2.5], colors=["#c6dbef", "#fcbba1", "#c7e9c0"])
for c in range(3):
    plt.scatter(*P2[Y2 == c].T, s=10, color=cols[c])
    w_c, b_c = W2[:2, c], W2[2, c]
    xs = np.linspace(-5, 5, 2)
    plt.plot(xs, -(w_c[0] * xs + b_c) / w_c[1], "--", color=cols[c], lw=1, label=f"score of class {c} = 0")
    plt.arrow(0, 0, *(1.5 * w_c / np.linalg.norm(w_c)), head_width=0.15, color=cols[c])
plt.xlim(-5, 5); plt.ylim(-3, 5.5); plt.legend(fontsize=7, loc="lower right"); plt.show()

In high dimensions the picture is the same, with hyperplanes instead of lines.

---
## 3. Hard cases for a linear classifier

Some arrangements of classes **can't be separated by flat walls**:
* **XOR**: class 1 in the first and third quadrants, class 2 in the second and fourth (Lecture 1's perceptron failure).
* **A ring**: class 1 where $1\leq\|x\|_2\leq2$, class 2 everywhere else.
* **Multiple modes**: class 1 in three separate blobs, class 2 everywhere else.

**Step 1: make the three datasets** (1,500 random points in a square, labeled by each rule).

In [ ]:
hard_rng = np.random.default_rng(2)
Pts = hard_rng.uniform(-3, 3, (1500, 2))
radius = np.linalg.norm(Pts, axis=1)
blob_centers = np.array([[-1.8, -1.5], [1.8, -1.2], [0, 1.9]])
dist_to_blobs = np.linalg.norm(Pts[:, None] - blob_centers[None], axis=2)   # (1500, 3)
cases = {
    "XOR (quadrants)":  (Pts[:, 0] * Pts[:, 1] > 0).astype(int),
    "ring 1 <= |x| <= 2": ((radius >= 1) & (radius <= 2)).astype(int),
    "three modes":       (dist_to_blobs.min(1) < 0.8).astype(int),
}
for name, yc in cases.items():
    print(f"{name:20s}: {yc.mean():.0%} of points are class 1")

**Step 2: a feature transform for each.** The fix (the pre-deep-learning recipe) is to compute **new features** in which the data *is* linearly separable, then train a linear classifier on those:
* **XOR:** add the product $x_1x_2$ (its sign *is* the class).
* **Ring:** use the radius $r = \|x\|$ and $r^2$ (the ring is "$r^2$ between 1 and 4", which is linear in $r$ and $r^2$).
* **Modes:** use "closeness to each blob center", $e^{-\text{distance}^2}$.

**[CS231n 2026]** shows the same idea with **polar coordinates** $(r, \theta)$.

In [ ]:
feature_maps = {
    "XOR (quadrants)":   lambda P: np.c_[P, P[:, 0] * P[:, 1]],
    "ring 1 <= |x| <= 2": lambda P: np.c_[np.linalg.norm(P, axis=1), np.linalg.norm(P, axis=1) ** 2],
    "three modes":       lambda P: np.exp(-np.linalg.norm(P[:, None] - blob_centers[None], axis=2) ** 2),
}

def standardize_with_bias(F):
    # Scale each feature to mean 0 / std 1 (so gradient descent behaves), then append the bias feature.
    return np.c_[(F - F.mean(0)) / F.std(0), np.ones(len(F))]

**Step 3: train a linear classifier on raw coordinates vs. on the transformed features.**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (name, yc) in zip(ax, cases.items()):
    X_raw = np.c_[Pts, np.ones(len(Pts))]
    acc_raw = np.mean(np.argmax(X_raw @ train_small(X_raw, yc, 2), 1) == yc)
    X_feat = standardize_with_bias(feature_maps[name](Pts))
    acc_feat = np.mean(np.argmax(X_feat @ train_small(X_feat, yc, 2), 1) == yc)
    majority = max(yc.mean(), 1 - yc.mean())
    print(f"{name:20s}: raw coordinates {acc_raw:.0%} (majority guess {majority:.0%}),  transformed features {acc_feat:.0%}")
    a.scatter(*Pts.T, c=yc, cmap="coolwarm", s=4); a.set_aspect("equal")
    a.set_title(f"{name}\nraw (x, y): {acc_raw:.0%} (always guessing the majority: {majority:.0%})\n"
                f"transformed features: {acc_feat:.0%}", fontsize=8)
plt.tight_layout(); plt.show()

**What to notice:** on raw coordinates the linear classifier does no better than always guessing the more common class. With the right features it's nearly perfect. Designing such features by hand was the pre-2012 approach (Lecture 1: SIFT, HOG); **neural networks learn the feature transform** (Lecture 5).

---
## 4. Choosing a good W: loss functions

To find good weights we need two things:
1. A **loss function** that measures how unhappy we are with the scores on the training data. (Also called an objective or cost function.)
2. A way to efficiently find the weights that **minimize** it (**optimization**, Lecture 4).

Given a dataset $\{(x_i, y_i)\}_{i=1}^N$ ($x_i$ an image, $y_i$ its integer label), the dataset loss is the **average of per-example losses**:
$$L = \frac1N\sum_{i=1}^NL_i\big(f(x_i, W), y_i\big)$$

### 4.1 The multiclass SVM loss
**Idea: the correct class's score should beat every other score by a margin (here, 1).** With scores $s = f(x_i, W)$:
$$L_i = \sum_{j\neq y_i}\max\big(0,\ s_j - s_{y_i} + 1\big)$$
Read it one wrong class $j$ at a time:
* If $s_{y_i}\geq s_j + 1$ (correct beats $j$ by at least 1): contributes **0**.
* Otherwise: contributes **how far short** it fell, $s_j - s_{y_i} + 1$.

The function $\max(0, 1 - \text{margin})$ is called the **hinge loss** (its graph looks like a door hinge).

**The slides' example**: three training images and their scores (correct class in bold):

| | cat image | car image | frog image |
|---|---|---|---|
| **cat** score | **3.2** | 1.3 | 2.2 |
| **car** score | 5.1 | **4.9** | 2.5 |
| **frog** score | −1.7 | 2.0 | **−3.1** |

**By hand, for the cat image** (correct class cat, score 3.2):

In [ ]:
s = {"cat": 3.2, "car": 5.1, "frog": -1.7}
term_car = max(0, s["car"] - s["cat"] + 1)          # max(0, 5.1 - 3.2 + 1)
term_frog = max(0, s["frog"] - s["cat"] + 1)        # max(0, -1.7 - 3.2 + 1)
print(f"car term:  max(0, 5.1 - 3.2 + 1) = {term_car:.1f}")
print(f"frog term: max(0, -1.7 - 3.2 + 1) = {term_frog:.1f}")
print(f"L_cat = {term_car + term_frog:.1f}")

**As a function, for all three images.** We compute every margin at once: subtract each row's correct-class score from the whole row, add 1, clip at 0, and zero out the correct class's own entry.

In [ ]:
S_example = np.array([[3.2, 5.1, -1.7],     # cat image: scores for (cat, car, frog)
                      [1.3, 4.9, 2.0],      # car image
                      [2.2, 2.5, -3.1]])    # frog image
y_example = np.array([0, 1, 2])             # correct classes: cat, car, frog

def svm_losses(S, y, margin=1.0, include_correct=False, squared=False):
    # Per-example multiclass SVM losses. S: (N, C) scores, y: (N,) labels.
    correct = S[np.arange(len(y)), y][:, None]          # (N, 1): each example's correct-class score
    m = np.maximum(0, S - correct + margin)             # every margin term at once
    if squared:
        m = m ** 2
    if not include_correct:
        m[np.arange(len(y)), y] = 0                     # skip j = y_i
    return m.sum(axis=1)

L_i = svm_losses(S_example, y_example)
for name, l in zip(["cat", "car", "frog"], L_i):
    print(f"L_{name} = {l:.1f}")
print(f"L = average = {L_i.mean():.2f}    (slides: 2.9, 0, 12.9 -> 5.27)")

**The lecture's questions, answered by computation.**

**Q1: What happens to the car image's loss if its scores change a little?**

In [ ]:
S_jiggled = S_example.copy(); S_jiggled[1] += np.array([0.2, -0.1, 0.3])
print(f"car image loss after jiggling: {svm_losses(S_jiggled, y_example)[1]:.1f}  -> still 0 (car already wins by more than 1)")

**Q2: What are the minimum and maximum possible loss?** Minimum **0** (every margin satisfied); maximum **∞** (a wrong score can be arbitrarily large).

**Q3: At the start of training, $W$ is small, so all scores ≈ 0. What's the loss?** Each of the $C - 1$ wrong classes contributes $\max(0, 0 - 0 + 1) = 1$, so $L\approx C - 1$. **A useful sanity check**: if your first loss isn't close to $C - 1$, something is wrong.

In [ ]:
q3_rng = np.random.default_rng(0)
S_init = 1e-3 * q3_rng.normal(size=(1000, 10)); y_init = q3_rng.integers(0, 10, 1000)
print(f"10 classes, scores ~ 0: average loss {svm_losses(S_init, y_init).mean():.3f}  (C - 1 = 9)")

**Q4: What if the sum also included $j = y_i$?** That adds $\max(0, s_{y_i} - s_{y_i} + 1) = 1$ to every example: the loss shifts by 1, and the best $W$ doesn't change.

**Q5: What if we used the mean instead of the sum?** The loss is divided by $C - 1$: again the same best $W$.

**Q6: What about the squared hinge $\max(0, \cdot)^2$?** A **different** loss: it punishes large violations much more, and small ones less.

In [ ]:
print(f"Q4 include j = y_i: {svm_losses(S_example, y_example, include_correct=True).mean():.2f} = {L_i.mean():.2f} + 1")
print(f"Q5 mean instead of sum: {L_i.mean() / 2:.2f}  (divided by C - 1 = 2)")
print(f"Q6 squared hinge: {svm_losses(S_example, y_example, squared=True).mean():.2f}  (a different loss)")

**Q7: Suppose we found a $W$ with $L = 0$. Is it unique?** No: $2W$ also has $L = 0$, since doubling every score doubles every margin, and satisfied margins stay satisfied. Checked on the car image:

In [ ]:
print(f"car image with W: loss {svm_losses(S_example, y_example)[1]:.1f};  with 2W (all scores doubled): {svm_losses(2 * S_example, y_example)[1]:.1f}")

**So how do we choose between $W$ and $2W$?** The data loss alone can't. That's the job of **regularization**.

---
## 5. Regularization: beyond fitting the training data

$$L(W) = \underbrace{\frac1N\sum_{i=1}^NL_i(f(x_i, W), y_i)}_{\text{data loss: fit the training data}} + \underbrace{\lambda R(W)}_{\text{regularization: don't fit it } \textit{too} \text{ well}}$$
$\lambda$ is the **regularization strength**, a hyperparameter. Simple choices of $R$:
* **L2:** $R(W) = \sum_{k,l}W_{k,l}^2$
* **L1:** $R(W) = \sum_{k,l}|W_{k,l}|$
* **Elastic net (L1 + L2):** $R(W) = \sum_{k,l}\beta W_{k,l}^2 + |W_{k,l}|$

More complex ones, for neural networks: **dropout, batch normalization, cutout, mixup, stochastic depth** (Lectures 10–11).

**Why regularize?**
* **Express preferences** among weights that fit equally well.
* **Make the model simpler**, so it works on new data. (**Occam's Razor**: among competing explanations, prefer the simplest; William of Ockham, 1285–1347 **[CS231n 2026]**.)
* **Improve optimization** by adding curvature (Lecture 4).

### 5.1 Expressing preferences
For $x = [1, 1, 1, 1]$, the weights $w_1 = [1, 0, 0, 0]$ and $w_2 = [0.25, 0.25, 0.25, 0.25]$ give the **same score** ($w_1^\top x = w_2^\top x = 1$). Which does each regularizer prefer?

In [ ]:
x4 = np.ones(4)
w1, w2 = np.array([1.0, 0, 0, 0]), np.full(4, 0.25)
print(f"scores: w1 . x = {w1 @ x4}, w2 . x = {w2 @ x4}")
print(f"L2: R(w1) = {np.sum(w1 ** 2):.4f}, R(w2) = {np.sum(w2 ** 2):.4f}  -> L2 prefers w2 (spread out)")
print(f"L1: R(w1) = {np.sum(np.abs(w1)):.4f}, R(w2) = {np.sum(np.abs(w2)):.4f}  -> a tie")

**What to notice:**
* **L2 prefers $w_2$**: it likes to **spread out** the weights, so the classifier uses all the evidence rather than relying on one input (more robust to noise in any single pixel).
* **L1 is indifferent here**: every nonnegative weight vector with the same total has the same L1 norm. In general, though, **L1 produces sparse weights** (many exactly zero). 🔎 Why: L2's pull on a weight, $\partial(w^2)/\partial w = 2w$, fades as the weight shrinks, so small weights never quite reach zero; L1's pull, $\text{sign}(w)$, stays the same size all the way down. You'll see the sparsity in §9.

### 5.2 Prefer simpler models
The slides' picture: a few noisy points, a wiggly function $f_1$ that passes through all of them, and a simpler $f_2$ that doesn't. **Regularization pushes against fitting the data too well, so we don't fit the noise.**

**Step 1: data.** 10 noisy points from a straight line $y = 0.8x$.

In [ ]:
poly_rng = np.random.default_rng(4)
x_pts = np.sort(poly_rng.uniform(-1, 1, 10))
y_pts = 0.8 * x_pts + poly_rng.normal(0, 0.15, 10)        # the truth is a line; the noise is random
x_grid = np.linspace(-1, 1, 200)

**Step 2: fit a degree-9 polynomial** (10 coefficients: enough to pass through all 10 points). The **ridge regression** solution, which minimizes $\|Aw - y\|^2 + \lambda\|w\|^2$ where $A$ holds the powers $x^9, \dots, x, 1$, has the closed form $w = (A^\top A + \lambda I)^{-1}A^\top y$.

In [ ]:
def polynomial_features(x, degree=9):
    # Columns x^degree, ..., x^1, x^0 (one row per point).
    return np.vander(x, degree + 1)

def ridge_fit(x, y, lam):
    # Coefficients minimizing ||A w - y||^2 + lam ||w||^2 (closed form).
    A = polynomial_features(x)
    return np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ y)

**Step 3: compare almost no regularization with a little.**

In [ ]:
plt.figure(figsize=(6, 3.5)); plt.scatter(x_pts, y_pts, color="k", zorder=3, label="training data")
for lam, col in [(1e-10, "r"), (1e-2, "b")]:
    w = ridge_fit(x_pts, y_pts, lam)
    fit_err = np.mean((polynomial_features(x_pts) @ w - y_pts) ** 2)
    true_err = np.mean((polynomial_features(x_grid) @ w - 0.8 * x_grid) ** 2)
    plt.plot(x_grid, polynomial_features(x_grid) @ w, col, label=f"lambda = {lam:g}")
    print(f"lambda = {lam:8g}: training error {fit_err:.4f},  error vs. the true line {true_err:.3f},  size of weights {np.linalg.norm(w):.1f}")
plt.ylim(-1.5, 1.5); plt.legend(fontsize=8); plt.show()

**What to notice:** without regularization, the fit hits every training point (nearly zero training error) with huge weights, and swings wildly between them. With a little L2, training error rises a bit, but the curve is far closer to the truth.

### 5.3 Regularization on the digits
Overfitting is worst with little data, so we train on only **150 images** and vary λ:

In [ ]:
small = np.arange(150)
print(f"{'lambda':>8s} {'train acc':>10s} {'val acc':>8s} {'||W||':>7s}")
for lam in [0.0, 1e-3, 1e-2, 1e-1, 1.0]:
    W_l = train_linear(softmax_loss, X_tr[small], y_tr[small], reg=lam, steps=3000)
    print(f"{lam:8g} {accuracy(W_l, X_tr[small], y_tr[small]):10.1%} {accuracy(W_l, X_va, y_va):8.1%} {np.linalg.norm(W_l[:64]):7.2f}")

**What to notice:** as λ grows, the weights shrink and training accuracy falls (the model is no longer allowed to fit every training image). Validation accuracy peaks at a moderate λ, and too much regularization underfits badly. λ is a hyperparameter, tuned on validation data (Lecture 2).

---
## 6. Softmax classifier (multinomial logistic regression)

The SVM treats scores as arbitrary numbers. **Softmax interprets scores as probabilities.** Three steps:
1. **Exponentiate** each score: $e^{s_k}$. All values become positive (*probabilities must be ≥ 0*).
2. **Normalize**: divide by their sum, so they add to 1 (*probabilities must sum to 1*):
$$P(Y = k\mid X = x_i) = \frac{e^{s_k}}{\sum_je^{s_j}}\qquad\text{(the softmax function)}$$
3. **Loss** = the **negative log-probability of the correct class**:
$$L_i = -\log P(Y = y_i\mid X = x_i)$$

The raw scores are called **logits** (unnormalized log-probabilities). Minimizing this loss is **maximum likelihood estimation**: choose the weights that make the observed labels as probable as possible.

**Step by step on the slides' cat image** (scores: cat 3.2, car 5.1, frog −1.7):

In [ ]:
s = S_example[0]                               # [3.2, 5.1, -1.7]
e = np.exp(s)
print("1. exponentiate:", e.round(2), "         (slides: 24.5, 164.0, 0.18)")
p = e / e.sum()
print("2. normalize:   ", p.round(2), "            (slides: 0.13, 0.87, 0.00)")
print(f"3. loss = -log(P(cat)) = -log({p[0]:.4f}) = {-np.log(p[0]):.2f}   (slides: 2.04, natural log)")

**The lecture's questions:**
* **Q1: min/max possible softmax loss?** Min **0** (when $P(\text{correct}) = 1$, which would need the correct score infinitely larger, so it's never reached); max **∞** (when $P(\text{correct})\to0$).
* **Q2: at initialization, all scores are about equal. What's the loss with $C$ classes?** Every probability is $1/C$, so $L_i = -\log(1/C) = \log C$: **for 10 classes, $\log 10\approx2.3$.** Another first-iteration sanity check:

In [ ]:
L_init, _ = softmax_loss(0.0001 * np.random.default_rng(0).normal(size=(65, 10)), X_tr, y_tr, 0.0)
print(f"softmax loss with tiny random weights: {L_init:.4f};  log(10) = {np.log(10):.4f}")

### 6.1 Cross-entropy and KL divergence
The softmax loss compares the **predicted** distribution $Q$ (the softmax probabilities) with the **target** distribution $P$ (all probability on the correct class: $[1, 0, 0]$ for the cat). Two standard measures of difference between distributions:
$$\text{KL divergence: } D_{KL}(P\,\|\,Q) = \sum_yP(y)\log\frac{P(y)}{Q(y)},\qquad\text{cross-entropy: } H(P, Q) = -\sum_yP(y)\log Q(y) = H(P) + D_{KL}(P\,\|\,Q)$$
For a one-hot target, the entropy $H(P) = 0$, so **cross-entropy = KL divergence = the softmax loss**; that's why it's also called the **cross-entropy loss**. With a "soft" target (🔎 used in label smoothing and distillation), $H(P) > 0$ and the identity $H(P, Q) = H(P) + D_{KL}$ still holds:

In [ ]:
def entropy(d):
    d = d[d > 0]; return float(abs(-np.sum(d * np.log(d))))   # abs() avoids printing -0.0
def kl_divergence(p_, q_):
    m = p_ > 0; return np.sum(p_[m] * np.log(p_[m] / q_[m]))
def cross_entropy(p_, q_):
    m = p_ > 0; return -np.sum(p_[m] * np.log(q_[m]))

Q = p                                                    # our predicted probabilities for the cat image
for name, P_t in [("one-hot target [1, 0, 0]", np.array([1.0, 0, 0])), ("soft target [0.7, 0.2, 0.1]", np.array([0.7, 0.2, 0.1]))]:
    print(f"{name:28s}: H(P) = {entropy(P_t):.4f}, KL = {kl_divergence(P_t, Q):.4f}, H + KL = {entropy(P_t) + kl_divergence(P_t, Q):.4f}, "
          f"cross-entropy = {cross_entropy(P_t, Q):.4f}")

### 6.2 🔎 Computing softmax safely
$e^s$ overflows for large scores ($e^{1000} = \infty$ in float64). Because softmax doesn't change when you subtract the same number from every score,
$$\frac{e^{s_k - m}}{\sum_je^{s_j - m}} = \frac{e^{-m}e^{s_k}}{e^{-m}\sum_je^{s_j}} = \frac{e^{s_k}}{\sum_je^{s_j}}$$
we subtract the **largest** score $m$ first, so every exponent is ≤ 0. Every real implementation does this (so does `softmax_loss` above).

In [ ]:
big = np.array([1000.0, 1001.0, 999.0])
with np.errstate(over="ignore", invalid="ignore"):
    naive = np.exp(big) / np.exp(big).sum()
stable = np.exp(big - big.max()) / np.exp(big - big.max()).sum()
print("naive softmax :", naive, "  <- overflow gives inf / inf = nan")
print("stable softmax:", stable.round(4))

---
## 7. Gradients of the losses, and checking them

To minimize a loss with gradient descent (Lecture 4), we need its **gradient** with respect to $W$: how the loss changes as each weight changes. With $W$ of shape $D\times C$ (column $w_j$ for class $j$) and scores $s_j = x_i^\top w_j$:

**Softmax.** With $p_j = \text{softmax}(s)_j$:
$$\frac{\partial L_i}{\partial s_j} = p_j - \mathbb{1}[j = y_i]\qquad\Longrightarrow\qquad\frac{\partial L_i}{\partial w_j} = \big(p_j - \mathbb{1}[j = y_i]\big)\,x_i$$
*Why:* $L_i = -s_{y_i} + \log\sum_ke^{s_k}$. The first term's derivative with respect to $s_j$ is $-1$ if $j = y_i$ (else 0); the second's is $e^{s_j}/\sum_ke^{s_k} = p_j$. This **"predicted probability minus target"** is one of the most important formulas in deep learning.

**Multiclass SVM.** Each **violated margin** (wrong class $j$ with $s_j - s_{y_i} + 1 > 0$) adds $+x_i$ to $\partial L_i/\partial w_j$ and $-x_i$ to $\partial L_i/\partial w_{y_i}$. (At the exact kink of the hinge we pick one side; in practice it never matters.)

**L2 regularization:** $\partial(\lambda\sum W^2)/\partial W = 2\lambda W$.

**Tiny check of the softmax gradient** on the cat image: probabilities [0.13, 0.87, 0.00], target [1, 0, 0], so $\partial L/\partial s = [0.13 - 1,\ 0.87,\ 0.00] = [-0.87, 0.87, 0.00]$: push the cat score up and the car score down.

In [ ]:
dL_ds = p.copy(); dL_ds[0] -= 1                # probabilities minus the one-hot target (correct class = cat)
print("dL/ds for the cat image:", dL_ds.round(3))
h = 1e-6                                        # numerical check, one score at a time
for j in range(3):
    s_plus = s.copy(); s_plus[j] += h
    numeric = (-np.log(np.exp(s_plus)[0] / np.exp(s_plus).sum()) - (-np.log(p[0]))) / h
    print(f"  score {j}: formula {dL_ds[j]:+.4f}, numerical {numeric:+.4f}")

**The SVM loss and gradient, written with loops** (easy to read, slow), exactly following the rules above:

In [ ]:
def svm_loss_naive(W, X, y, reg):
    # Multiclass SVM loss + L2, and its gradient, with explicit loops. Returns (loss, dL/dW).
    dW = np.zeros_like(W); loss = 0.0
    N, C = X.shape[0], W.shape[1]
    for i in range(N):
        s = X[i] @ W                                   # scores for example i
        for j in range(C):
            if j == y[i]:
                continue
            margin = s[j] - s[y[i]] + 1
            if margin > 0:                             # violated margin
                loss += margin
                dW[:, j] += X[i]                       # raise in loss as s_j grows
                dW[:, y[i]] -= X[i]                    # fall in loss as the correct score grows
    return loss / N + reg * np.sum(W * W), dW / N + 2 * reg * W

**The vectorized version** does the same with matrices. The trick: build a matrix $G$ ($N\times C$) with a 1 for every violated margin, and in each row's correct-class column put **minus the number of violations in that row**. Then $\partial L/\partial W = X^\top G/N$. Tiny look at $G$ for the slides' three images:

In [ ]:
def svm_loss_vectorized(W, X, y, reg):
    # Same as svm_loss_naive, without loops.
    N = X.shape[0]
    S = X @ W
    margins = np.maximum(0, S - S[np.arange(N), y][:, None] + 1)
    margins[np.arange(N), y] = 0
    loss = margins.sum() / N + reg * np.sum(W * W)
    G = (margins > 0).astype(float)                  # 1 for each violated margin
    G[np.arange(N), y] = -G.sum(axis=1)              # correct class: minus the number of violations
    return loss, X.T @ G / N + 2 * reg * W

margins = np.maximum(0, S_example - S_example[np.arange(3), y_example][:, None] + 1); margins[np.arange(3), y_example] = 0
G = (margins > 0).astype(float); G[np.arange(3), y_example] = -G.sum(1)
print("G for the three images (rows: cat, car, frog image):\n", G)

Read $G$: the cat image violated one margin (car), the car image none, the frog image two (cat and car).

**The softmax loss with loops**, for comparison with the vectorized `softmax_loss` from §2.2:

In [ ]:
def softmax_loss_naive(W, X, y, reg):
    # Softmax loss + L2, and its gradient, with explicit loops. Returns (loss, dL/dW).
    dW = np.zeros_like(W); loss = 0.0
    for i in range(X.shape[0]):
        s = X[i] @ W
        s = s - s.max()                                          # numerical safety
        p_i = np.exp(s) / np.exp(s).sum()
        loss -= np.log(p_i[y[i]])
        for j in range(W.shape[1]):
            dW[:, j] += (p_i[j] - (j == y[i])) * X[i]            # (probability - target) * x
    return loss / X.shape[0] + reg * np.sum(W * W), dW / X.shape[0] + 2 * reg * W

### 7.1 Gradient checking
Gradient code is easy to get wrong, so we **always compare it with a numerical estimate** (Lecture 4 explains the details): nudge one weight by $\pm h$, see how the loss changes, and compare with the analytic gradient using the **relative error** $\frac{|g_{\text{analytic}} - g_{\text{numerical}}|}{\max(|g_{\text{analytic}}|, |g_{\text{numerical}}|)}$:
$$g_{\text{numerical}} = \frac{L(W + h) - L(W - h)}{2h}$$

In [ ]:
def grad_check(f, W, n_checks=10, h=1e-5, seed=0):
    # Compare the analytic gradient of f (which returns (loss, grad)) with numerical estimates at n random entries.
    # Returns the largest relative error found.
    r = np.random.default_rng(seed)
    _, g = f(W)
    worst = 0.0
    for _ in range(n_checks):
        ix = tuple(r.integers(0, d) for d in W.shape)          # a random entry of W
        old = W[ix]
        W[ix] = old + h; loss_plus = f(W)[0]
        W[ix] = old - h; loss_minus = f(W)[0]
        W[ix] = old                                            # restore
        numerical = (loss_plus - loss_minus) / (2 * h)
        worst = max(worst, abs(numerical - g[ix]) / max(abs(numerical), abs(g[ix]), 1e-12))
    return worst

**Run the checks**: the loop and vectorized versions must agree with each other, and with the numerical gradient.

In [ ]:
W_chk = 0.01 * np.random.default_rng(0).normal(size=(65, 10))
Xc, yc = X_tr[:50], y_tr[:50]
for name, f_loop, f_vec in [("SVM", svm_loss_naive, svm_loss_vectorized), ("softmax", softmax_loss_naive, softmax_loss)]:
    l_loop, g_loop = f_loop(W_chk, Xc, yc, 0.1)
    l_vec, g_vec = f_vec(W_chk, Xc, yc, 0.1)
    rel = grad_check(lambda W_: f_vec(W_, Xc, yc, 0.1), W_chk)
    print(f"{name:8s}: loop vs vectorized: loss diff {abs(l_loop - l_vec):.1e}, gradient diff {np.abs(g_loop - g_vec).max():.1e};  "
          f"numerical check: max relative error {rel:.1e}")

Relative errors around $10^{-7}$ or below mean the gradient is right. And the speed difference:

In [ ]:
t0 = time.time(); svm_loss_naive(W_chk, X_tr, y_tr, 0.1); t_loop = time.time() - t0
t0 = time.time(); svm_loss_vectorized(W_chk, X_tr, y_tr, 0.1); t_vec = time.time() - t0
print(f"1,000 images: loops {t_loop * 1000:.0f} ms, vectorized {t_vec * 1000:.1f} ms")

---
## 8. Softmax vs. SVM

Both are linear classifiers; they differ in the loss. The difference shows when the scores are **already good**. Three examples where the correct class (first) is already ahead:

In [ ]:
def softmax_losses(S, y):
    # Per-example softmax loss, computed as log1p(sum_{j != y} exp(s_j - s_y)) so tiny losses stay accurate.
    diff = S - S[np.arange(len(y)), y][:, None]
    diff[np.arange(len(y)), y] = -np.inf               # exclude the correct class
    return np.log1p(np.exp(diff).sum(axis=1))

S3 = np.array([[10, -2, 3], [10, 9, 9], [10, -100, -100]], float); y3 = np.zeros(3, int)
S3_better = S3.copy(); S3_better[:, 0] = 20            # raise the correct score from 10 to 20
print(f"{'scores':22s} {'SVM':>6s} {'softmax':>10s}   | correct score -> 20:  {'SVM':>4s} {'softmax':>9s}")
for row, a, b_, c_, d_ in zip(S3, svm_losses(S3, y3), softmax_losses(S3, y3), svm_losses(S3_better, y3), softmax_losses(S3_better, y3)):
    print(f"{str(row):22s} {a:6.2f} {b_:10.2e}   | {'':21s} {c_:4.1f} {d_:9.2e}")

**What to notice:**
* **SVM:** all three have **zero loss**. Once the margins are satisfied, it doesn't care; raising the correct score to 20 changes nothing.
* **Softmax:** the loss is never exactly zero, and **keeps shrinking** as the correct score grows: it always wants more separation.

In practice the two reach similar accuracy (§9). Softmax is far more common today because its outputs are **probabilities**, which plug into everything else (confidence estimates, cross-entropy training of networks, language models).

### 8.1 🔎 Binary softmax is logistic regression
With two classes, softmax depends only on the **difference** of the scores:
$$P(y = 1) = \frac{e^{s_1}}{e^{s_0} + e^{s_1}} = \frac{1}{1 + e^{-(s_1 - s_0)}} = \sigma(s_1 - s_0)$$
(divide top and bottom by $e^{s_1}$). That's the **sigmoid** of logistic regression.

In [ ]:
r8 = np.random.default_rng(0); s0, s1 = r8.normal(size=5), r8.normal(size=5)
print("2-class softmax == sigmoid of the difference:",
      np.allclose(np.exp(s1) / (np.exp(s0) + np.exp(s1)), 1 / (1 + np.exp(-(s1 - s0)))))

---
## 9. Training both on real digits

The full recipe: preprocessed data → scores → loss + regularization → gradient → update. We try a few (λ, learning rate) pairs, pick the best on **validation**, and report **test** accuracy once.

In [ ]:
best = {}
for name, loss_fn in [("SVM", svm_loss_vectorized), ("softmax", softmax_loss)]:
    candidates = []
    for lam in [1e-4, 1e-3, 1e-2]:
        for lr in [0.05, 0.2]:
            W_c = train_linear(loss_fn, X_tr, y_tr, reg=lam, lr=lr, steps=2000)
            candidates.append((accuracy(W_c, X_va, y_va), lam, lr, W_c))
    val_acc, lam, lr, W_best = max(candidates, key=lambda c: c[0])
    best[name] = W_best
    print(f"{name:8s}: best lambda {lam:g}, learning rate {lr}: validation {val_acc:.1%}  ->  TEST {accuracy(W_best, X_te, y_te):.1%}")

**🔎 L1 makes weights sparse.** Train with an L1 penalty (its gradient is $\lambda\,\text{sign}(W)$) and with L2 of the same strength, and count near-zero weights:

In [ ]:
def softmax_loss_l1(W, X, y, reg):
    # Softmax loss with an L1 penalty instead of L2.
    loss, dW = softmax_loss(W, X, y, 0.0)
    return loss + reg * np.abs(W).sum(), dW + reg * np.sign(W)

W_l1 = train_linear(softmax_loss_l1, X_tr, y_tr, reg=3e-3, lr=0.2, steps=3000)
W_l2 = train_linear(softmax_loss, X_tr, y_tr, reg=3e-3, lr=0.2, steps=3000)
near_zero = lambda W_: np.mean(np.abs(W_[:64]) < 0.01)
print(f"weights near zero (|w| < 0.01): L1 {near_zero(W_l1):.0%}, L2 {near_zero(W_l2):.0%};  "
      f"validation accuracy: L1 {accuracy(W_l1, X_va, y_va):.1%}, L2 {accuracy(W_l2, X_va, y_va):.1%}")

In [ ]:
fig, ax = plt.subplots(3, 10, figsize=(14, 4.4))
for row, (name, W_) in enumerate([("SVM", best["SVM"]), ("softmax", best["softmax"]), ("softmax + L1", W_l1)]):
    for c in range(10):
        ax[row, c].imshow(W_[:64, c].reshape(8, 8), cmap="RdBu_r"); ax[row, c].axis("off")
    ax[row, 0].set_title(name, fontsize=8, loc="left")
plt.suptitle("templates from each loss (bottom: L1 regularization -> sparse templates)", fontsize=9); plt.show()

**What to notice:** both losses learn similar templates and similar accuracy; L1's templates use far fewer pixels. About 95–96% is roughly the ceiling for a *linear* classifier on these digits; Lecture 5's two-layer network does better.

---
## 10. PyTorch: linear classifiers on CIFAR-10 (run in Colab)
* A linear classifier is `nn.Linear(3072, 10)`.
* `nn.CrossEntropyLoss` is the softmax loss; it takes **logits** and applies log-softmax internally (stably).
* `nn.MultiMarginLoss` is the multiclass SVM loss.
* L2 regularization is usually passed to the optimizer as `weight_decay`.

Expect roughly **40%** test accuracy for a linear softmax classifier on CIFAR-10 raw pixels (vs. ~35–39% for kNN, 10% chance).

In [ ]:
# [PyTorch] Needs: pip install torch torchvision (downloads CIFAR-10, ~170MB).
import torch, torch.nn as nn, torchvision
tr_ds = torchvision.datasets.CIFAR10("./data", train=True, download=True)
te_ds = torchvision.datasets.CIFAR10("./data", train=False, download=True)
Xtr_t = torch.tensor(tr_ds.data, dtype=torch.float32).reshape(50000, -1) / 255; ytr_t = torch.tensor(tr_ds.targets)
Xte_t = torch.tensor(te_ds.data, dtype=torch.float32).reshape(10000, -1) / 255; yte_t = torch.tensor(te_ds.targets)
mu = Xtr_t.mean(0); Xtr_t, Xte_t = Xtr_t - mu, Xte_t - mu                       # subtract the mean training image
for name, loss_fn in [("softmax", nn.CrossEntropyLoss()), ("SVM", nn.MultiMarginLoss())]:
    torch.manual_seed(0)
    model = nn.Linear(3072, 10)
    opt = torch.optim.SGD(model.parameters(), lr=1e-2 if name == "softmax" else 1e-3, weight_decay=1e-4)
    for step in range(3000):
        idx = torch.randint(0, 50000, (256,))
        loss = loss_fn(model(Xtr_t[idx]), ytr_t[idx])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        print(f"{name}: CIFAR-10 test accuracy {(model(Xte_t).argmax(1) == yte_t).float().mean().item():.1%}")

---
## 11. Summary

| Topic | Takeaway |
|---|---|
| Parametric approach | $f(x, W) = Wx + b$: the training data is summarized in $W$ (30,730 numbers for CIFAR-10 vs. 153.6M) |
| Bias trick | Append 1 to $x$ and $b$ to $W$ |
| Algebraic view | Scores are weighted sums of pixels |
| Visual view | One template per class; can't represent several modes (the two-headed horse) |
| Geometric view | Flat walls (hyperplanes); fails on XOR, rings, multiple modes unless features are transformed |
| Loss function | Average of per-example losses: how unhappy we are with the scores |
| Multiclass SVM | $\sum_{j\neq y_i}\max(0, s_j - s_{y_i} + 1)$; ≈ C − 1 at the start; zero once margins hold |
| Regularization | Data loss + λR(W); L2 spreads weights, L1 zeroes them; prefer simpler models |
| Softmax | $P = e^{s_k}/\sum_je^{s_j}$, $L_i = -\log P(y_i)$; ≈ log C at the start; never fully satisfied |
| Cross-entropy | $H(P, Q) = H(P) + D_{KL}(P\|Q)$; equals the softmax loss for one-hot targets |
| Gradients | Softmax: (probabilities − one-hot) × x; SVM: ±x per violated margin; always gradient-check |

### Check yourself
<details><summary>1. W has shape 10 × 3072 for CIFAR-10. What does row 3 represent, and how can you see it?</summary>

The template for class 3: one weight per pixel and color channel. Reshape it to 32 × 32 × 3 and display it as an image (after rescaling to 0–255).
</details>

<details><summary>2. Why is the SVM loss about C − 1 at initialization, and the softmax loss about log C?</summary>

With all scores ≈ 0, each of the C − 1 wrong classes contributes max(0, 0 − 0 + 1) = 1 to the SVM loss. For softmax, every probability is 1/C, so the loss is −log(1/C) = log C.
</details>

<details><summary>3. If W gives zero SVM loss, why isn't it unique, and what decides between W and 2W?</summary>

Scaling W scales every margin, so satisfied margins stay satisfied: 2W also has zero loss. Regularization breaks the tie (L2 prefers the smaller W).
</details>

<details><summary>4. What's the gradient of the softmax loss with respect to the scores?</summary>

∂L/∂s_j = p_j − 1[j = y_i]: the predicted probability minus the one-hot target.
</details>

<details><summary>5. Why subtract the maximum score before computing softmax?</summary>

e^s overflows for large s. Softmax is unchanged when the same constant is subtracted from all scores, so subtracting the max gives the same answer with every exponent ≤ 0.
</details>

<details><summary>6. Why does L1 regularization give sparse weights while L2 doesn't?</summary>

L2's gradient 2w shrinks along with the weight, so its push toward zero fades; L1's gradient sign(w) keeps the same size until the weight reaches zero.
</details>

### Exercises
1. Show the CIFAR-10 templates from the PyTorch cell (reshape each row of `model.weight` to 32×32×3). Find the two-headed horse and the blue planes.
2. Implement the **squared hinge** loss and its gradient; gradient-check it; compare its accuracy with the standard hinge.
3. Implement **elastic net** regularization; plot sparsity vs. validation accuracy as β varies.
4. Add **label smoothing** to the softmax loss (target $1 - \epsilon$ on the correct class, $\epsilon/(C - 1)$ on the others). Derive and gradient-check the new gradient.
5. Train a linear classifier on §3's hard cases using **200 random features** $\cos(\omega^\top x + b)$ (random $\omega$, $b$). Which cases does it solve?

### References
* CS231n course notes, *Linear classification: SVM, Softmax* (cs231n.github.io/linear-classify).
* Weston & Watkins 1999; Crammer & Singer 2001 (multiclass SVMs).
* Bishop, *Pattern Recognition and Machine Learning*, §4.3; Goodfellow, Bengio & Courville, *Deep Learning*, §5.2, §6.2.2.
* Tibshirani 1996 (lasso); Zou & Hastie 2005 (elastic net).

**Next:** Lecture 4, Optimization.